# KIBA on four Kaggle accounts, two T4s each

The DAVIS audit is finished; KIBA is the replication. This notebook trains the **18 KIBA
cells** the paper needs — the two published models we audit plus the accuracy anchor,
at {random, cold_drug}, three seeds — split across four accounts so that **both T4s of
every account do the same number of hours**.

## Scope, and why these 18 cells

| level | why | KIBA's held-out set |
|---|---|---|
| random | replicates the one cell of sixteen that survived Holm on DAVIS | 2,057 drugs / 228 targets |
| cold_drug | the axis DAVIS cannot support: it holds out 13 drugs, KIBA holds out **422** | 422 drugs |

Three deliberate exclusions, each with its reason, all of which have to be stated in the
paper rather than left to a reader to notice:

* **`cold_target` and `cold_pair`** — KIBA holds out only 45 targets against DAVIS's 88,
  so they would add a weaker copy of a level DAVIS already covers, for ~63 GPU-h.
* **ColdSite-DTI** — our own model, cut to fit the compute (−27 GPU-h). Its DAVIS verdict
  is unambiguous (at chance against annotated residues at all four levels, the whole
  bootstrap interval at random lying *below* chance), so the replication is aimed at the
  two **published** models whose claims the audit is actually about. The paper must say
  that our model is audited on one dataset where the published ones are audited on two.
* **The explanation-side analyses** — readouts, integrated gradients, per-pair drug
  contacts — stay DAVIS-only.

## What each account does

| account | GPU0 | GPU1 | wall time | commits |
|---|---|---|---|---|
| **A** | MolTrans random s1 → HAT cold_drug s2 | MolTrans random s2 → HAT cold_drug s3 | 16.7 h | two |
| **B** | HAT random s1 → HAT random s3 | HAT random s2 → HAT cold_drug s1 | 14.6 h | two |
| **C** | MolTrans random s3 → 2 DeepDTA | MolTrans cold_drug s1 → 2 DeepDTA | 9.6 h | **one** |
| **D** | MolTrans cold_drug s2 → DeepDTA random s3 | MolTrans cold_drug s3 → DeepDTA cold_drug s1 | 9.5 h | **one** |

Every account's two queues are equal to within 0.0 h. Totals: **100.8 GPU-h at 36 epochs,
69.6 at the 25-epoch minimum**. C and D fit inside a single 11-hour commit; A and B need a
second one.

## What to do

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **On**.
2. Set `ACCOUNT` in section 1 to this account's letter. **Nothing else.**
3. **Save Version -> Save & Run All (Commit)**.
4. Only if it stops before finishing: download the output, make it a private dataset,
   attach it, set `RESTORE_FROM = '/kaggle/input'`, and run again. Finished cells are
   skipped and an interrupted cell continues from its last finished epoch.

**No two accounts may use the same letter.** Section 1 prints the cells it owns; if two
accounts print the same cell, one of them is wasting a day.


## 1. Settings — set ACCOUNT, nothing else

In [1]:
# ============================================================================
# SETTINGS
# ============================================================================

ACCOUNT = 'B'            # 'A' | 'B' | 'C' | 'D' -- this account's share. Nothing else.

RESTORE_FROM = None      # second commit onward: '/kaggle/input' (searches every input)

# ============================================================================
# Everything below is the plan. Read it; do not edit it.
# ============================================================================

DATASET = 'kiba'
TASK = 'binary'
BRANCH = 'main'          # resume and --amp were merged into main on 2026-09-14
AMP = True               # validated on DAVIS: results/amp_validation_davis.md
LEVELS = ['random', 'cold_drug']
SEEDS = [1, 2, 3]
# ColdSite-DTI is deliberately absent: see the header. HOURS still carries it so the
# cost model stays complete if it is ever added back.
KIBA_MODELS = ['deepdta', 'hyperattentiondti', 'moltrans']

ACCOUNTS = {
    # Account: {GPU: [(model, split, seed), ...]}, most expensive cell first. Hours are
    # the measured T4 projections (results/speed_test_kiba_t4.md), mixed precision, at
    # the 36-epoch median of the DAVIS histories.
    'A': {                                  # GPU0 16.7 h | GPU1 16.7 h -> two commits
        0: [('moltrans', 'random', 1), ('hyperattentiondti', 'cold_drug', 2)],
        1: [('moltrans', 'random', 2), ('hyperattentiondti', 'cold_drug', 3)],
    },
    'B': {                                  # GPU0 14.6 h | GPU1 14.6 h -> two commits
        0: [('hyperattentiondti', 'random', 1), ('hyperattentiondti', 'random', 3)],
        1: [('hyperattentiondti', 'random', 2), ('hyperattentiondti', 'cold_drug', 1)],
    },
    'C': {                                  # GPU0  9.6 h | GPU1  9.6 h -> one commit
        0: [('moltrans', 'random', 3),
            ('deepdta', 'random', 1), ('deepdta', 'cold_drug', 2)],
        1: [('moltrans', 'cold_drug', 1),
            ('deepdta', 'random', 2), ('deepdta', 'cold_drug', 3)],
    },
    'D': {                                  # GPU0  9.5 h | GPU1  9.5 h -> one commit
        0: [('moltrans', 'cold_drug', 2), ('deepdta', 'random', 3)],
        1: [('moltrans', 'cold_drug', 3), ('deepdta', 'cold_drug', 1)],
    },
}

# Hours per cell on one T4, mixed precision, at 36 epochs / at the 25-epoch minimum
# (results/speed_test_kiba_t4.md; random and cold_drug have 82,778 and 83,807 training
# rows, so one figure covers both).
HOURS = {'deepdta': (0.1, 0.1), 'coldsite_dti': (4.5, 3.1),
         'hyperattentiondti': (7.3, 5.0), 'moltrans': (9.4, 6.5)}

# Minutes per epoch on one T4 with mixed precision, same source. The runner compares the
# epochs it actually times against these, so a GPU that is slower than the benchmark
# shows up in the first half hour instead of at hour eleven.
MIN_PER_EPOCH = {'deepdta': 0.2, 'coldsite_dti': 7.5,
                 'hyperattentiondti': 12.1, 'moltrans': 15.7}
EPOCHS_MIN, EPOCHS_TYPICAL = 25, 36     # early-stopping floor, and the DAVIS median

assert ACCOUNT in ACCOUNTS, f"ACCOUNT must be one of {sorted(ACCOUNTS)}, got {ACCOUNT!r}"

# The whole grid, and this account's share of it.
EVERY_CELL = [(m, lv, s) for m in KIBA_MODELS for lv in LEVELS for s in SEEDS]
MY_CELLS = ACCOUNTS[ACCOUNT]

# --- the plan is checked here, on Kaggle, not only in the test suite -------------------
_flat = [c for gpu in sorted(ACCOUNTS) for q in ACCOUNTS[gpu].values() for c in q]
assert len(_flat) == len(set(_flat)) == 18, (
    f'the plan covers {len(_flat)} cells, {len(set(_flat))} of them distinct -- '
    'a cell listed twice would be trained twice')
assert sorted(_flat) == sorted(EVERY_CELL), (
    'the plan is not exactly the 18 cells: '
    f'missing {sorted(set(EVERY_CELL) - set(_flat))}, '
    f'unexpected {sorted(set(_flat) - set(EVERY_CELL))}')
for _name, _queues in ACCOUNTS.items():
    _loads = [sum(HOURS[m][0] for m, _lv, _s in q) for q in _queues.values()]
    assert max(_loads) - min(_loads) <= 0.2, (
        f'account {_name}: its GPUs differ by {max(_loads) - min(_loads):.1f} h')

MODELS = sorted({m for q in MY_CELLS.values() for m, _lv, _s in q})   # for later cells
TOTAL_CELLS = sum(len(q) for q in MY_CELLS.values())

print(f'KIBA, account {ACCOUNT} of {len(ACCOUNTS)} -- no other account may use '
      f'{ACCOUNT!r}')
print(f'{TOTAL_CELLS} cells, mixed precision, branch {BRANCH}')
print()
for _gpu in sorted(MY_CELLS):
    _hi = sum(HOURS[m][0] for m, _lv, _s in MY_CELLS[_gpu])
    _lo = sum(HOURS[m][1] for m, _lv, _s in MY_CELLS[_gpu])
    print(f'GPU{_gpu}: {_lo:.1f}-{_hi:.1f} h')
    for _m, _lv, _s in MY_CELLS[_gpu]:
        print(f'   {_m:18s} {_lv:10s} seed {_s}   {HOURS[_m][1]:.1f}-{HOURS[_m][0]:.1f} h')
print()
_wall_lo = max(sum(HOURS[m][1] for m, _l, _s in q) for q in MY_CELLS.values())
_wall_hi = max(sum(HOURS[m][0] for m, _l, _s in q) for q in MY_CELLS.values())
# A commit self-stops at 11 h; leave a little room for the clone and the splits.
_verdict = ('should finish in ONE commit' if _wall_hi <= 10.5 else
            'fits one commit at the 25-epoch floor, two at the median'
            if _wall_lo <= 10.5 else 'expect two commits')
print(f'wall time = the slower GPU: {_wall_lo:.1f}-{_wall_hi:.1f} h -> {_verdict}')


KIBA, account B of 4 -- no other account may use 'B'
4 cells, mixed precision, branch main

GPU0: 10.0-14.6 h
   hyperattentiondti  random     seed 1   5.0-7.3 h
   hyperattentiondti  random     seed 3   5.0-7.3 h
GPU1: 10.0-14.6 h
   hyperattentiondti  random     seed 2   5.0-7.3 h
   hyperattentiondti  cold_drug  seed 1   5.0-7.3 h

wall time = the slower GPU: 10.0-14.6 h -> fits one commit at the 25-epoch floor, two at the median


## 2. Check the GPU(s)

In [2]:
import time
START = time.time()          # the 11-hour self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)

# Measured peak memory on a 1000-residue protein (STATUS.md), from the DAVIS 36-grid:
#   ColdSite-DTI  batch 64 -> 8.7 GB, 16 -> 2.3 GB
#   HyperAttentionDTI  batch 32 -> ~5 GB; 8 x accum 4 is the same effective batch
#   DeepDTA  batch 256 -> 0.7 GB
#   MolTrans  not yet profiled on a full split -- stays at the vendored batch size (16)
#   rather than guessing a larger one on a bigger GPU.
big = torch.cuda.get_device_properties(0).total_memory / 1e9 >= 14
COLDSITE_BATCH = 64 if big else 16
HAT_BATCH, HAT_ACCUM = (32, 1) if big else (8, 4)
DEEPDTA_BATCH = 256
MOLTRANS_BATCH = 16
print(f'batches: ColdSite {COLDSITE_BATCH}, HAT {HAT_BATCH}x{HAT_ACCUM}, '
      f'DeepDTA {DEEPDTA_BATCH}, MolTrans {MOLTRANS_BATCH}')


GPU 0: Tesla T4, 15.6 GB
GPU 1: Tesla T4, 15.6 GB
torch  : 2.10.0+cu128
batches: ColdSite 64, HAT 32x1, DeepDTA 256, MolTrans 16


## 3. Clone the repo

`BRANCH` is `main`: epoch-level resume (`src/model/resume.py`) and `--amp`
(`src/model/precision.py`) were merged there on 2026-09-14. The old KIBA plan said
`kiba-resume`; that branch is now **behind** main and must not be used.


In [3]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch {BRANCH} {REPO} {SRC}
os.chdir(SRC)
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip install -q tabulate subword-nmt

import importlib, src.model.dataset as _ds
importlib.reload(_ds)
assert hasattr(_ds, 'BINARY_THRESHOLD'), (
    'This checkout predates the binary-label fix -- ColdSite-DTI would crash. '
    'Re-run this cell so git pull fetches the latest commit.')
if 'moltrans' in MODELS:
    assert os.path.exists('src/model/train_moltrans.py'), (
        'This checkout has no src/model/train_moltrans.py -- push that commit to '
        'origin/main before running MolTrans here, then re-run this cell.')

# Without these two, a cell that passes the 11-hour mark restarts from epoch 1 for
# ever, and --amp is not available at all. Both are on main since 2026-09-14.
for _needed in ('src/model/resume.py', 'src/model/precision.py'):
    assert os.path.exists(_needed), (
        f'{_needed} is missing from branch {BRANCH!r}. KIBA cannot be trained without it: '
        'a cell longer than one 11-hour commit would never finish. Check that BRANCH is '
        "'main' and that this clone pulled.")

RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
!git rev-parse --abbrev-ref HEAD
print('results ->', RESULTS)


Cloning into '/kaggle/working/ColdSite-DTI_New'...
remote: Enumerating objects: 1507, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 1507 (delta 24), reused 26 (delta 11), pack-reused 1455 (from 1)
Receiving objects: 100% (1507/1507), 185.46 MiB | 27.15 MiB/s, done.
Resolving deltas: 100% (920/920), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/Mahim56207/ColdSite-DTI_New
 * branch            main       -> FETCH_HEAD
Already up to date.

837e9d6 (HEAD -> main, origin/main, origin/HEAD) KIBA: MolTrans trains in full precision -- float16 makes it diverge
main
results -> /kaggle/working/results


## 4. Fetch the DeepDTA source files

`build_splits` needs these for both datasets regardless of which models this run trains.

In [4]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.load_data


davis: 30056 measured pairs, 68 unique drugs, 442 unique targets, Y range [5.000, 10.796]
kiba: 118254 measured pairs, 2111 unique drugs, 229 unique targets, Y range [0.000, 17.200]


## 5. Build the splits — and verify they match

KIBA's four splits are rebuilt from the source files and checked against the row counts
this project recorded. A silent change here would train every cell on the wrong data.


In [5]:
!python -m src.data.build_splits 2>&1 | grep -E 'kiba|leakage'

import pandas as pd

# Recorded 2026-09-14 from data/splits/kiba on the Mac. The two levels this notebook
# trains are checked exactly; the other two are printed for completeness.
EXPECTED = {
    'random':      (82778, 11825, 23651),
    'cold_drug':   (83807, 12073, 22374),
    'cold_target': (85452, 10701, 22101),
    'cold_pair':   (58041,  1334,  4375),
}
POSITIVE_RATE = {'random': 0.209, 'cold_drug': 0.219}   # test set, threshold 12.1

from src.model.dataset import BINARY_THRESHOLD
assert BINARY_THRESHOLD['kiba'] == 12.1, BINARY_THRESHOLD
print(f"binary threshold: KIBA score >= {BINARY_THRESHOLD['kiba']}")

for level, expected in EXPECTED.items():
    sizes = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{level}/{part}.csv'))
                  for part in ('train', 'valid', 'test'))
    mark = 'OK' if sizes == expected else f'MISMATCH, expected {expected}'
    print(f'{level:12s} {str(sizes):28s} {mark}')
    if level in LEVELS:
        assert sizes == expected, (
            f'{level}: split sizes {sizes} do not match the recorded {expected}. '
            'Do not train on this -- the cells would not be comparable with anything.')

# A wrong threshold or a mislabelled column would show up here and nowhere else until
# the results made no sense.
for level in LEVELS:
    test = pd.read_csv(f'data/splits/{DATASET}/{level}/test.csv')
    rate = (test['Y'] >= BINARY_THRESHOLD[DATASET]).mean()
    assert abs(rate - POSITIVE_RATE[level]) < 0.01, (
        f'{level}: {rate:.1%} of test rows are positive, expected '
        f'{POSITIVE_RATE[level]:.1%} -- check the threshold and the Y column')
    print(f'{level:12s} test positives {rate:.1%}  OK')
print('\nsplits and labels match the record.')


cold_drug: no leakage across train/valid/test -- OK
cold_target: no leakage across train/valid/test -- OK
cold_pair: no leakage across train/valid/test -- OK
=== Building splits for kiba ===
Saved data/splits/kiba/random/  train=82778  valid=11825  test=23651
Saved data/splits/kiba/cold_drug/  train=83807  valid=12073  test=22374
Saved data/splits/kiba/cold_target/  train=85452  valid=10701  test=22101
Saved data/splits/kiba/cold_pair/  train=58041  valid=1334  test=4375
cold_drug: no leakage across train/valid/test -- OK
cold_target: no leakage across train/valid/test -- OK
cold_pair: no leakage across train/valid/test -- OK
binary threshold: KIBA score >= 12.1
random       (82778, 11825, 23651)        OK
cold_drug    (83807, 12073, 22374)        OK
cold_target  (85452, 10701, 22101)        OK
cold_pair    (58041, 1334, 4375)          OK
random       test positives 20.9%  OK
cold_drug    test positives 21.9%  OK

splits and labels match the record.


## 6. Restore from a previous commit

Set `RESTORE_FROM = '/kaggle/input'` in section 1 from the second commit onward. This
copies finished cells (so they are skipped) **and** `_resume.pt` files, which is how an
interrupted cell continues from its last finished epoch instead of starting again.


In [6]:
import shutil, glob

# '/kaggle/input' -- searches every attached input. Kaggle does not always mount a
# dataset at /kaggle/input/<name> (2026-09-12: that path failed; this one restored 65).
if RESTORE_FROM:
    assert os.path.isdir(RESTORE_FROM), f'not a directory: {RESTORE_FROM}'
    copied = skipped = 0
    for src in glob.glob(f'{RESTORE_FROM}/**/*', recursive=True):
        name = os.path.basename(src)
        if not name.endswith(('.pt', '_results.json', '_history.json')):
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f'restored {copied} file(s), left {skipped} already present')
else:
    print('RESTORE_FROM is None -- starting from an empty results folder.')


RESTORE_FROM is None -- starting from an empty results folder.


## 7. The runner

One queue per GPU, exactly as section 1 printed. The command builders are the ones the
DAVIS grid used, with `--amp` added for every cell.


In [7]:
import subprocess, threading, glob, json, re
from src.model.checkpoint_naming import checkpoint_path, results_path, run_tag

DEADLINE = START + 11 * 3600
STATUS_EVERY = 600          # seconds between STATUS lines

assert N_GPU >= len(MY_CELLS), (
    f'this plan has {len(MY_CELLS)} queues but Kaggle gave {N_GPU} GPU(s). '
    'Settings -> Accelerator -> GPU T4 x2, then re-run: on one GPU this account would '
    'take twice as long and overrun its quota.')

MODEL_NAME = {'src.model.train_deepdta': 'DeepDTA', 'src.model.run_grid': 'ColdSite',
              'src.model.train_hyperattentiondti': 'HAT', 'src.model.train_moltrans': 'MolTrans'}
MODEL_KEY = {'src.model.train_deepdta': 'deepdta', 'src.model.run_grid': 'coldsite_dti',
             'src.model.train_hyperattentiondti': 'hyperattentiondti',
             'src.model.train_moltrans': 'moltrans'}
HEADER = re.compile(r'^\w+/(\w+)/seed(\d+)\s*$')          # run_grid: a cell starts
SKIPPED = re.compile(r'^\[skip\] \w+/(\w+)/seed(\d+)')     # run_grid: a cell was done
EPOCH = re.compile(r'^\s*(?:epoch|Epoch)\s+(\d+)')
SAVED = re.compile(r'Saved -> (\S+_results\.json)')


def hours_left():
    return (DEADLINE - time.time()) / 3600


def note_epoch(w, number, now):
    """Record an epoch boundary and keep a mean seconds-per-epoch.

    Called for every line that looks like an epoch header. The baselines print
    "epoch 1 train batch 7/10 ..." for every batch, so the same number arrives many
    times and only a CHANGE is a boundary. The first epoch of a cell also carries the
    split encoding (KIBA is 83,000 rows), so it is the starting mark but its own
    duration is never counted in the rate.
    """
    if number != w['last_epoch']:
        gap = number - w['last_epoch'] if w['last_epoch'] else 0
        if gap > 0 and w['last_epoch_at'] is not None:
            # Divide by the gap: a resumed cell's first reported epoch is not 1, and one
            # elapsed stretch may cover several epochs. Weighted mean, so a stretch of
            # three epochs counts three times as much as a single one.
            per = (now - w['last_epoch_at']) / gap
            n = w['timed_epochs']
            w['sec_per_epoch'] = (per if not n
                                  else (w['sec_per_epoch'] * n + per * gap) / (n + gap))
            w['timed_epochs'] = n + gap
        w['last_epoch'], w['last_epoch_at'] = number, now
    w['epoch'] = str(number)
    return w


def eta(w, now, deadline):
    """Two lines about one GPU: the measured rate, and when it finishes.

    `w` is that GPU's live state. Returns [] until a rate exists -- claiming an ETA from
    a single epoch would mean quoting the split-encoding time as the epoch time.
    """
    rate = w.get('sec_per_epoch')
    key, index = w.get('key'), w.get('cell', 0)
    if not rate or not key:
        return []
    predicted = MIN_PER_EPOCH[key] * 60
    drift = rate / predicted if predicted else float('nan')
    done = int(w.get('epoch') or 0)
    lines = [f"      {rate / 60:.1f} min/epoch measured, {predicted / 60:.1f} projected "
             f"(x{drift:.2f}) over {w.get('timed_epochs', 0)} epoch(s)"]

    # this cell, if it stops at the floor or at the DAVIS median
    at = []
    for label, total in (('min', EPOCHS_MIN), ('median', EPOCHS_TYPICAL)):
        left = max(total - done, 0) * rate
        at.append(f"{label} {time.strftime('%H:%M', time.localtime(now + left))}"
                  f" (+{left / 3600:.1f} h)")
    lines.append(f"      this cell ends: {' | '.join(at)}")

    # the rest of this GPU's queue, rescaled by the drift we are actually seeing
    queued = ORDERED.get(str(w.get('gpu')), [])
    remaining = queued[index:]                      # cells not started yet
    if remaining:
        rest = sum(HOURS[m][0] for m, _lv, _s in remaining) * drift
        this_cell = max(EPOCHS_TYPICAL - done, 0) * rate / 3600
        total_left = rest + this_cell
        verdict = ('inside this commit' if now + total_left * 3600 <= deadline
                   else f'needs about {int(total_left / 11) + 1} more commit(s)')
        lines.append(f"      queue: {len(remaining)} cell(s) after this one, "
                     f"~{total_left:.1f} h left at the median -> {verdict}")
    else:
        this_cell = max(EPOCHS_TYPICAL - done, 0) * rate / 3600
        verdict = ('inside this commit' if now + this_cell * 3600 <= deadline
                   else 'needs another commit')
        lines.append(f"      last cell of this queue, ~{this_cell:.1f} h left at the "
                     f"median -> {verdict}")
    return lines


def cells_done():
    """This account's finished cells. Counting a MODELS x LEVELS x SEEDS product would
    count cells another account owns and report progress that is not ours."""
    done = 0
    for queue in MY_CELLS.values():
        for model, split, seed in queue:
            tag = run_tag(DATASET, split, TASK, seed)
            if os.path.exists(results_path(RESULTS, tag, model=model)):
                done += 1
    return done


def same_as_other_seed(results_file):
    """Another seed of this model and split with exactly the same test metrics means the
    seed never reached training -- MolTrans's vendored import reseeded torch with 1, so
    its three DAVIS seeds were one run three times (2026-09-13)."""
    try:
        mine = json.load(open(results_file))['test_metrics']
        for other in glob.glob(re.sub(r'_seed\d+', '_seed[0-9]', results_file)):
            if other != results_file and json.load(open(other))['test_metrics'] == mine:
                return (f'SEEDS IDENTICAL: same test metrics as {os.path.basename(other)} '
                        f'-- is --seed reaching training? Stop and check before going on.')
    except Exception:
        return None
    return None

def _arg(cmd, flag):
    return cmd[cmd.index(flag) + 1] if flag in cmd else '-'


def _cells_in(cmd):
    if cmd[3] == 'src.model.run_grid':
        return len(_arg(cmd, '--splits').split(',')) * len(_arg(cmd, '--seeds').split(','))
    return 1


def run_parallel(queues, label):
    """queues: {gpu: [command, ...]}. Returns True if the deadline cut it short."""
    current, state = {}, {'deadline': False}
    where = {g: {'model': '-', 'split': '-', 'seed': '-', 'epoch': '-', 'cell': 0,
                 'total': sum(_cells_in(c) for c in q), 'gpu': g, 'key': None,
                 'sec_per_epoch': None, 'timed_epochs': 0, 'last_epoch': None,
                 'last_epoch_at': None}
             for g, q in queues.items()}

    def tag(g):
        w = where[g]
        return f"[GPU{g} {w['model']} {w['split']} s{w['seed']} · cell {w['cell']}/{w['total']}]"

    def begin_cell(g, split, seed):
        where[g].update(split=split, seed=seed, epoch='-', sec_per_epoch=None,
                        timed_epochs=0, last_epoch=None, last_epoch_at=None)
        where[g]['cell'] += 1

    def worker(gpu, commands):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for cmd in commands:
                if time.time() > DEADLINE:
                    return
                module = cmd[3]
                where[gpu]['model'] = MODEL_NAME.get(module, module)
                where[gpu]['key'] = MODEL_KEY.get(module)
                if module != 'src.model.run_grid':          # one command = one cell
                    begin_cell(gpu, _arg(cmd, '--split'), _arg(cmd, '--seed'))
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                current[gpu] = proc
                for line in proc.stdout:
                    if module == 'src.model.run_grid':      # follow run_grid's own cells
                        m = HEADER.match(line) or SKIPPED.match(line)
                        if m:
                            begin_cell(gpu, m.group(1), m.group(2))
                    m = EPOCH.match(line)
                    if m:
                        note_epoch(where[gpu], int(m.group(1)), time.time())
                    print(f'{tag(gpu)} {line}', end='', flush=True)
                    log.write(line)
                    log.flush()
                    m = SAVED.search(line)
                    if m:
                        try:
                            auc = json.load(open(m.group(1)))['test_metrics'].get('auroc')
                            auc = f'{auc:.4f}'
                        except Exception:
                            auc = '?'
                        print(f'  ✓ {tag(gpu)} finished -- test AUROC {auc}   '
                              f'[{cells_done()}/{TOTAL_CELLS} cells complete]', flush=True)
                        warning = same_as_other_seed(m.group(1))
                        if warning:
                            print(f'  !! {tag(gpu)} {warning}', flush=True)
                code_ = proc.wait()
                if code_ != 0 and not state['deadline']:
                    print(f'{tag(gpu)} !! exited {code_}: {" ".join(cmd[3:])}', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    last_status = 0.0
    while any(t.is_alive() for t in threads):
        if time.time() - last_status >= STATUS_EVERY:
            last_status = time.time()
            now_ = time.time()
            print(f"\n=== STATUS {time.strftime('%H:%M')} | "
                  f"{cells_done()}/{TOTAL_CELLS} cells complete | "
                  f"{(now_ - START) / 3600:.1f} h in, {hours_left():.1f} h before the "
                  f"stop ===", flush=True)
            for g, w in sorted(where.items()):
                print(f"   GPU{g}: {w['model']} {w['split']} s{w['seed']} "
                      f"cell {w['cell']}/{w['total']} epoch {w['epoch']}", flush=True)
                for line in eta(w, now_, DEADLINE):
                    print(line, flush=True)
            print(flush=True)
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** 11-hour mark: stopping so this commit can save its output. '
                  'Unfinished cells continue from their last finished epoch next commit. ***\n', flush=True)
            for proc in list(current.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(15)
    return state['deadline']


AMP_FLAG = ['--amp'] if AMP else []      # same value for every cell of this run


def deepdta_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_deepdta',
            '--split-dir', f'data/splits/{DATASET}/{split}', '--dataset', DATASET,
            '--split', split, '--task', 'binary', '--seed', str(seed),
            '--batch-size', str(DEEPDTA_BATCH), '--min-epochs', '10', '--epochs', '100',
            '--patience', '10',     # DeepDTA's own; DAVIS was trained with it
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done', *AMP_FLAG]


def coldsite_cmd(split, seed):
    # ColdSite-DTI has no single-cell entry point of its own, so it goes through run_grid
    # with one split and one seed -- one cell per command, like the other three.
    # patience is not a train.py flag; ColdSite-DTI's is fixed at 15 in run_training.
    return ['python', '-u', '-m', 'src.model.run_grid', '--datasets', DATASET,
            '--splits', split, '--seeds', str(seed),
            '--task', 'binary', '--epochs', '100', '--min-epochs', '10',
            '--batch-size', str(COLDSITE_BATCH), '--results-dir', RESULTS, *AMP_FLAG]


def hat_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_hyperattentiondti',
            '--split-dir', f'data/splits/{DATASET}/{split}', '--dataset', DATASET,
            '--split', split, '--seed', str(seed),
            '--batch-size', str(HAT_BATCH), '--accum-steps', str(HAT_ACCUM),
            '--patience', '15',
            '--min-epochs', '10', '--epochs', '100',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done', *AMP_FLAG]


def moltrans_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_moltrans',
            '--split-dir', f'data/splits/{DATASET}/{split}', '--dataset', DATASET,
            '--split', split, '--seed', str(seed),
            '--batch-size', str(MOLTRANS_BATCH), '--min-epochs', '10', '--epochs', '100',
            '--patience', '15',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done', *AMP_FLAG]


BUILDER = {'deepdta': deepdta_cmd, 'coldsite_dti': coldsite_cmd,
           'hyperattentiondti': hat_cmd, 'moltrans': moltrans_cmd}

# Most expensive cell FIRST in each queue. Two reasons, both measured:
#  * a resume file is as big as the model's optimiser state -- MolTrans's is 754 MB
#    (its checkpoint is 251 MB), against 28 MB for HyperAttentionDTI and 7 MB for
#    ColdSite-DTI. Running MolTrans first means it finishes inside this commit at the
#    36-epoch median (9.4 h < 11 h) and the cell the stop interrupts is a small one, so
#    the dataset you upload between commits is megabytes rather than gigabytes.
#  * it puts the riskiest cell in the first hour, where a problem is still cheap.
QUEUES = {str(gpu): [BUILDER[m](lv, s)
                     for m, lv, s in sorted(cells, key=lambda c: -HOURS[c[0]][0])]
          for gpu, cells in sorted(MY_CELLS.items())}

# Same order as QUEUES, so the runner can name the cell it is on and add up what is
# left in the queue.
ORDERED = {str(gpu): sorted(cells, key=lambda c: -HOURS[c[0]][0])
           for gpu, cells in sorted(MY_CELLS.items())}

# The commands are built from the plan, so this is the last chance to see a cell that
# belongs to another account.
for gpu, cells in sorted(MY_CELLS.items()):
    print(f'GPU {gpu}: in this order')
    for m, lv, s in sorted(cells, key=lambda c: -HOURS[c[0]][0]):
        print(f'   {m:18s} {lv:10s} seed {s}   {HOURS[m][1]:.1f}-{HOURS[m][0]:.1f} h')
print(f'{hours_left():.1f} h left before the self-stop')


GPU 0: in this order
   hyperattentiondti  random     seed 1   5.0-7.3 h
   hyperattentiondti  random     seed 3   5.0-7.3 h
GPU 1: in this order
   hyperattentiondti  random     seed 2   5.0-7.3 h
   hyperattentiondti  cold_drug  seed 1   5.0-7.3 h
11.0 h left before the self-stop


## 8. Launch

In [8]:
if hours_left() < 0.5:
    raise SystemExit('less than 30 minutes before the self-stop -- not worth starting')

cut = run_parallel(QUEUES, f'kiba_{ACCOUNT}')
print()
print(f'{cells_done()}/{TOTAL_CELLS} of account {ACCOUNT}\'s cells are complete')
if cut:
    print('CUT SHORT by the 11-hour stop. Download the output, make it a dataset, attach '
          "it, set RESTORE_FROM = '/kaggle/input' in section 1, and run again: finished "
          'cells are skipped and an interrupted cell continues from its last epoch.')
else:
    print('every cell of this account finished')



=== STATUS 14:22 | 0/4 cells complete | 0.0 h in, 11.0 h before the stop ===
   GPU0: HAT random s1 cell 1/2 epoch -
   GPU1: HAT random s2 cell 1/2 epoch -

[GPU0 HAT random s1 · cell 1/2]   train  82,778 pairs  21.0% positive  <- data/splits/kiba/random/train.csv
[GPU1 HAT random s2 · cell 1/2]   train  82,778 pairs  21.0% positive  <- data/splits/kiba/random/train.csv
[GPU0 HAT random s1 · cell 1/2]   valid  11,825 pairs  21.1% positive  <- data/splits/kiba/random/valid.csv
[GPU1 HAT random s2 · cell 1/2]   valid  11,825 pairs  21.1% positive  <- data/splits/kiba/random/valid.csv
[GPU0 HAT random s1 · cell 1/2]   test   23,651 pairs  20.9% positive  <- data/splits/kiba/random/test.csv
[GPU1 HAT random s2 · cell 1/2]   test   23,651 pairs  20.9% positive  <- data/splits/kiba/random/test.csv
[GPU0 HAT random s1 · cell 1/2] 
[GPU0 HAT random s1 · cell 1/2]   device        cuda (Tesla T4, 14.6 GB)
[GPU0 HAT random s1 · cell 1/2]   micro-batch   32 x 1 accum = effective 32 (vendored: 32

## 9. What landed

Test AUROC per cell. A KIBA `random` cell should land around 0.85-0.90 — far from 0.5
(not learning) and far from 0.99 (leakage). Two seeds of one cell with *identical*
metrics means `--seed` never reached training, which is the MolTrans bug from 2026-09-13;
the runner also warns about it as it happens.


In [9]:
import glob
import json

rows = []
for gpu, cells_ in sorted(MY_CELLS.items()):
    for model, split, seed in cells_:
        tag = run_tag(DATASET, split, TASK, seed)
        path = results_path(RESULTS, tag, model=model)
        if not os.path.exists(path):
            rows.append((gpu, model, split, seed, None, None, 'not finished'))
            continue
        blob = json.load(open(path))
        metrics = blob.get('test_metrics', {})
        rows.append((gpu, model, split, seed, metrics.get('auroc'), metrics.get('auprc'),
                     f"epoch {blob.get('best_epoch', '?')}"))

print(f'| GPU | model | split | seed | test AUROC | AUPRC | note |')
print(f'|---|---|---|---|---|---|---|')
for gpu, model, split, seed, auroc, auprc, note in rows:
    a = f'{auroc:.4f}' if auroc is not None else '—'
    p = f'{auprc:.4f}' if auprc is not None else '—'
    print(f'| {gpu} | {model} | {split} | {seed} | {a} | {p} | {note} |')

done = [r for r in rows if r[4] is not None]
print(f'\n{len(done)}/{len(rows)} cells finished')
for _gpu, model, split, seed, auroc, _p, _n in done:
    if not 0.6 <= auroc <= 0.98:
        print(f'!! {model} {split} s{seed}: AUROC {auroc:.4f} is outside the believable '
              f'0.60-0.98 band for KIBA -- check it before using this cell')

# Identical metrics across seeds of the same cell = --seed never reached training.
seen = {}
for _gpu, model, split, seed, auroc, auprc, _n in done:
    key = (model, split, auroc, auprc)
    if key in seen:
        print(f'!! SEEDS IDENTICAL: {model} {split} seeds {seen[key]} and {seed} have the '
              f'same test metrics. Do not use them; check that --seed reaches training.')
    seen[key] = seed


| GPU | model | split | seed | test AUROC | AUPRC | note |
|---|---|---|---|---|---|---|
| 0 | hyperattentiondti | random | 1 | 0.9335 | 0.8304 | epoch 14 |
| 0 | hyperattentiondti | random | 3 | — | — | not finished |
| 1 | hyperattentiondti | random | 2 | 0.9334 | 0.8294 | epoch 14 |
| 1 | hyperattentiondti | cold_drug | 1 | — | — | not finished |

2/4 cells finished


## 10. Take the results with you

Two zips. `results` is small and holds everything the analysis needs; `resume` holds the
`_resume.pt` files, which are large and are only needed if a cell was cut short. Upload
both to this account's private dataset if anything is unfinished; otherwise `results`
alone is enough.


In [10]:
import subprocess

RES_ZIP = f'{WORK}/kiba_{ACCOUNT}_results.zip'
RESUME_ZIP = f'{WORK}/kiba_{ACCOUNT}_resume.zip'

finished = [p for p in glob.glob(f'{RESULTS}/*')
            if not os.path.basename(p).endswith('_resume.pt')]
resumes = glob.glob(f'{RESULTS}/*_resume.pt')

for zip_path, paths in ((RES_ZIP, finished), (RESUME_ZIP, resumes)):
    if not paths:
        print(f'{os.path.basename(zip_path)}: nothing to pack')
        continue
    if os.path.exists(zip_path):
        os.remove(zip_path)
    subprocess.run(['zip', '-q', '-j', zip_path, *paths], check=True)
    size = os.path.getsize(zip_path) / 1e9
    print(f'{os.path.basename(zip_path)}: {len(paths)} file(s), {size:.2f} GB')

print()
print('Download from the Output panel. If any cell is unfinished, upload BOTH zips to '
      f"this account's private dataset, attach it, and set RESTORE_FROM = '/kaggle/input'.")


kiba_B_results.zip: 6 file(s), 0.03 GB
kiba_B_resume.zip: 2 file(s), 0.05 GB

Download from the Output panel. If any cell is unfinished, upload BOTH zips to this account's private dataset, attach it, and set RESTORE_FROM = '/kaggle/input'.
